In [ ]:
# Libraries
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import Image, display

from matplotlib.ticker import FuncFormatter

In [ ]:
# 
df = pd.read_excel("data.xlsx")
df.head()

# Data Cleaning
1. Date Handling
2. Duplication (merging)

In [ ]:
# Preserve original values
df["Value_Original"] = df["Value"]

# Campaign month
df["Campaign_Month"] = pd.to_datetime(
    df["Month"],
    format="%b-%y",
    errors="coerce"
)

# Identify Excel serial dates (including numeric strings)
serial_mask = (
    df["Value"].astype(str)
    .str.strip()
    .str.fullmatch(r"\d{5}(?:\.0)?")
    .fillna(False)
)
numeric = pd.to_numeric(
    df["Value"].where(serial_mask),
    errors="coerce"
)

# Convert serial values to Excel dates
excel_dates = pd.to_datetime(
    numeric,
    unit="D",
    origin="1899-12-30",
    errors="coerce"
)

# Parse the already formatted dates
text_dates = pd.to_datetime(
    df["Value"].where(~serial_mask),
    dayfirst=True,
    errors="coerce"
)

# Standard conversion
df["Date_Original"] = excel_dates.fillna(text_dates)

# Create corrected date
df["Date_Corrected"] = df["Date_Original"].copy()

# Swap day and month for identified serial dates
swap_mask = (
    serial_mask
    & excel_dates.notna()
    & excel_dates.dt.day.between(1, 12)
)

corrected = pd.to_datetime({
    "year": excel_dates.loc[swap_mask].dt.year,
    "month": excel_dates.loc[swap_mask].dt.day,
    "day": excel_dates.loc[swap_mask].dt.month
})

df.loc[swap_mask, "Date_Corrected"] = corrected

# Flag dates inconsistent with the campaign month
month_mismatch = (
    df["Date_Corrected"].dt.to_period("M")
    != df["Campaign_Month"].dt.to_period("M")
)

df["Date_Status"] = "Valid"
df.loc[serial_mask, "Date_Status"] = "Corrected"
df.loc[month_mismatch, "Date_Status"] = "Review"
df.loc[df["Date_Corrected"].isna(), "Date_Status"] = "Invalid"

# Sort by corrected date
df = df.sort_values("Date_Corrected").reset_index(drop=True)

# Inspect results
print(
    df[
        [
            "Month",
            "Value_Original",
            "Date_Original",
            "Date_Corrected",
            "Date_Status"
        ]
    ].to_string(index=False)
)

In [ ]:
# Backup original dataset
df_original = df.copy()

# Columns identifying duplicates
group_cols = ["Campaign", "Platform", "Date_Corrected"]

# Columns to SUM
sum_cols = [
    "Impressions",
    "Visits",
    "Incremental Visits",
    "First Time Deposit",
    "Incremental First Time Deposit",
    "Bet",
    "Incremental Bet",
    "Deposit",
    "Incremental Deposit"
]

# Identify duplicate groups
duplicate_mask = df.duplicated(
    subset=group_cols,
    keep=False
)

# Separate duplicates and non-duplicates
df_duplicates = df.loc[duplicate_mask].copy()
df_unique = df.loc[~duplicate_mask].copy()

# Merge duplicate records
aggregation = {
    col: "first"
    for col in df.columns
    if col not in group_cols + sum_cols
}

aggregation.update({
    col: "sum" for col in sum_cols
})

df_merged = (
    df_duplicates
    .groupby(group_cols, as_index=False, dropna=False)
    .agg(aggregation)
)

# Replace duplicates with merged rows
df_clean = pd.concat(
    [df_unique, df_merged],
    ignore_index=True
)

# Recalculate rates
rate_mapping = {
    "Visit Rate": "Visits",
    "Incremental Visit Rate": "Incremental Visits",
    "First Time Deposit Rate": "First Time Deposit",
    "Incremental First Time Deposit Rate": "Incremental First Time Deposit",
    "Bet Rate": "Bet",
    "Incremental Bet Rate": "Incremental Bet",
    "Deposit Rate": "Deposit",
    "Incremental Deposit Rate": "Incremental Deposit"
}

for rate_col, metric_col in rate_mapping.items():
    df_clean[rate_col] = np.where(
        df_clean["Impressions"] > 0,
        df_clean[metric_col] / df_clean["Impressions"] * 100,
        np.nan
    )

# Sort and preserve original column order
df_clean = (
    df_clean[df.columns]
    .sort_values(group_cols)
    .reset_index(drop=True)
)

# Check results
print("Original rows:", len(df_original))
print("Duplicate rows identified:", duplicate_mask.sum())
print("Cleaned rows:", len(df_clean))

display(df_clean.head())